# Визуализация записанных демонстраций

Этот ноутбук воспроизводит записанный эпизод в MuJoCo и накладывает поверх симуляции изображения,
которые были сохранены в датасете.

Что показывает окно:
- основная симуляция воспроизводит действие из датасета;
- в правом верхнем углу отображается изображение основной камеры из датасета;
- в правом нижнем углу отображается изображение второй камеры из датасета.


In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from lerobot.common.datasets.utils import write_json, serialize_dict
import numpy as np

# Настройки датасета
REPO_NAME = "so101_pnp"
ROOT = "./demo_data"

# Если хочешь открыть примерный датасет, можно поменять ROOT на './demo_data_example'
dataset = LeRobotDataset(REPO_NAME, root=ROOT)

print(f"Датасет загружен: {REPO_NAME}")
print(f"Корневая папка: {dataset.root}")
print(f"Количество эпизодов: {dataset.num_episodes}")
print(f"Количество кадров: {len(dataset)}")


## Загрузка одного эпизода

In [ ]:
import torch

class EpisodeSampler(torch.utils.data.Sampler):
    """
    Сэмплер для одного выбранного эпизода.
    """
    def __init__(self, dataset: LeRobotDataset, episode_index: int):
        from_idx = dataset.episode_data_index["from"][episode_index].item()
        to_idx = dataset.episode_data_index["to"][episode_index].item()
        self.frame_ids = range(from_idx, to_idx)

    def __iter__(self):
        return iter(self.frame_ids)

    def __len__(self) -> int:
        return len(self.frame_ids)


In [ ]:
# Выбери индекс эпизода, который хочешь воспроизвести
episode_index = 0

episode_sampler = EpisodeSampler(dataset, episode_index)
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=1,
    batch_size=1,
    sampler=episode_sampler,
)

print(f"Выбран эпизод: {episode_index}")
print(f"Количество кадров в эпизоде: {len(episode_sampler)}")


## Воспроизведение датасета в симуляции

In [ ]:
from mujoco_env.y_env import SimpleEnv

xml_path = "./asset/example_scene_y.xml"

# Для воспроизведения действий из датасета используем режим абсолютных углов суставов
PnPEnv = SimpleEnv(
    xml_path,
    action_type="joint_angle",
    state_type="joint_angle",
)


In [ ]:
step = 0
iter_dataloader = iter(dataloader)
PnPEnv.reset()

while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()

    if PnPEnv.env.loop_every(HZ=20):
        try:
            data = next(iter_dataloader)
        except StopIteration:
            # Начинаем эпизод заново
            iter_dataloader = iter(dataloader)
            PnPEnv.reset()
            step = 0
            continue

        if step == 0:
            # Ставим кружку и тарелку в стартовые позиции этого эпизода
            PnPEnv.set_obj_pose(data["obj_init"][0, :3], data["obj_init"][0, 3:])

        # Действие из датасета: [shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll, gripper]
        action = data["action"][0].cpu().numpy().astype(np.float32)
        _ = PnPEnv.step(action)

        # Подготовка изображений из датасета для overlay
        rgb_agent = data["observation.image"][0].cpu().numpy()
        rgb_wrist = data["observation.wrist_image"][0].cpu().numpy()

        # Перевод из диапазона [0, 1] в uint8
        rgb_agent = np.clip(rgb_agent * 255.0, 0, 255).astype(np.uint8)
        rgb_wrist = np.clip(rgb_wrist * 255.0, 0, 255).astype(np.uint8)

        # Из формата (C, H, W) в (H, W, C), если нужно
        if rgb_agent.ndim == 3 and rgb_agent.shape[0] in (1, 3, 4):
            rgb_agent = np.transpose(rgb_agent, (1, 2, 0))
        if rgb_wrist.ndim == 3 and rgb_wrist.shape[0] in (1, 3, 4):
            rgb_wrist = np.transpose(rgb_wrist, (1, 2, 0))

        PnPEnv.rgb_agent = rgb_agent
        PnPEnv.rgb_ego = rgb_wrist
        PnPEnv.rgb_side = np.zeros((480, 640, 3), dtype=np.uint8)

        PnPEnv.render()
        step += 1

        if step >= len(episode_sampler):
            # После конца эпизода начинаем его заново
            iter_dataloader = iter(dataloader)
            PnPEnv.reset()
            step = 0


In [ ]:
# Закрыть окно симуляции
PnPEnv.env.close_viewer()


### [Необязательно] Сохранить `stats.json` для совместимости с другими версиями

In [ ]:
stats = dataset.meta.stats
path_stats = dataset.root / "meta" / "stats.json"
stats = serialize_dict(stats)

write_json(stats, path_stats)
print(f"Файл сохранён: {path_stats}")


Готово. Теперь можно:
- выбрать другой `episode_index`;
- открыть окно MuJoCo и посмотреть воспроизведение;
- сохранить `stats.json`, если он нужен для других скриптов.
